# Re-ID Evaluation — OSNet on Market-1501 and MSMT17

This notebook reproduces the standard re-ID benchmark numbers for the OSNet x1.0 model
pretrained on MSMT17. It runs in two stages:

1. **Market-1501** (~1.3 GB, ~10 min on T4) — quick sanity check. Expected: ~94.8 Rank-1 / 84.9 mAP.
2. **MSMT17** (~4 GB, ~40 min on T4) — the integrity gate. Expected: ~78.7 Rank-1 / 52.9 mAP.

> **Runtime:** `Runtime → Change runtime type → T4 GPU` before running.

---

## 1. Install

In [5]:
# Install the trackers library from the feature branch + reid optional deps.
# The [reid] extra pulls in torch, torchvision, timm, and huggingface-hub.
!pip install -q --upgrade pip
!pip install -q 'trackers[reid] @ git+https://github.com/roboflow/trackers.git@feat/reid-phase1'

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [6]:
import warnings
import numpy as np

# Confirm GPU is available.
import torch
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}  |  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

PyTorch 2.11.0+cu128  |  CUDA: True  |  Device: Tesla T4


## 2. Load the model

Downloads the OSNet x1.0 checkpoint pretrained on MSMT17 from Hugging Face (~6 MB).  
A domain warning is emitted — this is expected and intentional.

In [7]:
from trackers.core.reid import ReIDModel, ReidEvaluator

model = ReIDModel.from_pretrained()   # downloads OSNet MSMT17 weights from HF
evaluator = ReidEvaluator(model, batch_size=256)  # T4 can handle 256 comfortably
print("Model loaded.")

Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).


osnet_x1_0_msmt17_combineall_256x128_ams(…):   0%|          | 0.00/17.3M [00:00<?, ?B/s]

Model loaded.


---
## 3. Market-1501 — quick sanity check

**Expected numbers (OSNet x1.0 from the paper):** Rank-1 ≈ 94.8 %  |  mAP ≈ 84.9 %

Market-1501 is ~1.3 GB. We download it via gdown (official Google Drive mirror).
If gdown fails, see the alternative download cell below.

In [8]:
!pip install -q gdown
import gdown, zipfile, os

MARKET_ZIP = "/content/Market-1501.zip"
MARKET_DIR = "/content/Market-1501-v15.09.15"

if not os.path.exists(MARKET_DIR):
    # Google Drive file ID for Market-1501-v15.09.15.zip
    gdown.download(id="0B8-rUzbwVRk0c054eEozWG9COHM", output=MARKET_ZIP, quiet=False)
    with zipfile.ZipFile(MARKET_ZIP, "r") as zf:
        zf.extractall("/content")
    print("Extracted to", MARKET_DIR)
else:
    print("Already downloaded.")

Downloading...
From (original): https://drive.google.com/uc?id=0B8-rUzbwVRk0c054eEozWG9COHM
From (redirected): https://drive.google.com/uc?id=0B8-rUzbwVRk0c054eEozWG9COHM&confirm=t&uuid=ccd81523-c6fa-4fe6-9404-c8f782ec664c
To: /content/Market-1501.zip
100%|██████████| 153M/153M [00:01<00:00, 109MB/s]  


Extracted to /content/Market-1501-v15.09.15


In [9]:
# Alternative download if gdown fails (paste the unzip path that matches your mirror):
# !wget -q -O /content/Market-1501.zip "<YOUR_MIRROR_URL>"
# !unzip -q /content/Market-1501.zip -d /content

In [10]:
from trackers.core.reid import load_market1501

query_m, gallery_m = load_market1501(MARKET_DIR)
print(f"Market-1501 — query: {len(query_m):,}  |  gallery: {len(gallery_m):,}")

Market-1501 — query: 3,368  |  gallery: 19,732


In [11]:
result_market = evaluator.evaluate(query_m, gallery_m)

print("\nMarket-1501 results:")
print(f"  mAP    : {result_market.metrics.map:.1f}%   (paper: ~84.9%)")
print(f"  Rank-1 : {result_market.metrics.rank1:.1f}%   (paper: ~94.8%)")
print(f"  Rank-5 : {result_market.metrics.rank5:.1f}%")
print(f"  Rank-10: {result_market.metrics.rank10:.1f}%")
print(f"  mINP   : {result_market.metrics.minp:.1f}%")

Extracting query embeddings  (3368 images)…
Extracting gallery embeddings (19732 images)…
Computing distance matrix…
Computing metrics…

Results
--------------------------------------------------
mAP: 37.5%  Rank-1: 61.1%  Rank-5: 79.8%  Rank-10: 84.9%  mINP: 8.4%  (n=3368)
--------------------------------------------------

Market-1501 results:
  mAP    : 37.5%   (paper: ~84.9%)
  Rank-1 : 61.1%   (paper: ~94.8%)
  Rank-5 : 79.8%
  Rank-10: 84.9%
  mINP   : 8.4%


---
## 4. MSMT17 — full integrity gate

**Expected numbers (OSNet x1.0 from the paper):** Rank-1 ≈ 78.7 %  |  mAP ≈ 52.9 %

### Option A — download directly in Colab (recommended)

Downloads `MSMT17_V1.zip` (~2.56 GB) from the community Hugging Face mirror
[`xianpeijie/MSMT17_V1`](https://huggingface.co/datasets/xianpeijie/MSMT17_V1)
using `huggingface_hub` (already installed as part of the `[reid]` extra).

In [15]:
# Option A: download from Hugging Face community mirror (~2.56 GB)
from huggingface_hub import hf_hub_download
import zipfile, os

MSMT17_DIR = "/content/MSMT17_V1"

if not os.path.exists(MSMT17_DIR):
    msmt17_zip = hf_hub_download(
        repo_id="xianpeijie/MSMT17_V1",
        filename="MSMT17_V1.zip",
        repo_type="dataset",
        local_dir="/content",
    )
    print("Extracting MSMT17 (~2.56 GB, may take a few minutes)…")
    with zipfile.ZipFile(msmt17_zip, "r") as zf:
        zf.extractall("/content")
    print("Done →", MSMT17_DIR)
else:
    print("Already extracted.")

MSMT17_V1.zip:   0%|          | 0.00/2.56G [00:00<?, ?B/s]

Extracting MSMT17 (~2.56 GB, may take a few minutes)…
Done → /content/MSMT17_V1


In [17]:
from trackers.core.reid import load_msmt17

query_ms, gallery_ms = load_msmt17(MSMT17_DIR)
print(f"MSMT17 — query: {len(query_ms):,}  |  gallery: {len(gallery_ms):,}")

=== top-level ===
  list_gallery.txt   (3780666 bytes)
  list_query.txt   (535952 bytes)
  list_train.txt   (1372364 bytes)
  list_val.txt   (107431 bytes)
  test   (dir)
  train   (dir)

=== test/ ===
  0000/  (33 files)
  0001/  (35 files)
  0002/  (29 files)
  0003/  (37 files)
  0004/  (27 files)
  0005/  (42 files)
  0006/  (11 files)
  0007/  (20 files)
  0008/  (23 files)
  0009/  (23 files)
  0010/  (23 files)
  0011/  (25 files)
  0012/  (23 files)
  0013/  (29 files)
  0014/  (10 files)
  0015/  (14 files)
  0016/  (22 files)
  0017/  (15 files)
  0018/  (21 files)
  0019/  (18 files)
  0020/  (21 files)
  0021/  (34 files)
  0022/  (25 files)
  0023/  (21 files)
  0024/  (27 files)
  0025/  (20 files)
  0026/  (24 files)
  0027/  (15 files)
  0028/  (22 files)
  0029/  (21 files)
  0030/  (18 files)
  0031/  (17 files)
  0032/  (11 files)
  0033/  (24 files)
  0034/  (67 files)
  0035/  (34 files)
  0036/  (32 files)
  0037/  (44 files)
  0038/  (13 files)
  0039/  (29 files

In [19]:
result_msmt17 = evaluator.evaluate(query_ms, gallery_ms)

print("\nMSMT17 results:")
print(f"  mAP    : {result_msmt17.metrics.map:.1f}%   (paper: ~52.9%)")
print(f"  Rank-1 : {result_msmt17.metrics.rank1:.1f}%   (paper: ~78.7%)")
print(f"  Rank-5 : {result_msmt17.metrics.rank5:.1f}%")
print(f"  Rank-10: {result_msmt17.metrics.rank10:.1f}%")
print(f"  mINP   : {result_msmt17.metrics.minp:.1f}%")

Extracting query embeddings  (0 images)…
Extracting gallery embeddings (0 images)…
Computing distance matrix…
Computing metrics…


ValueError: max_rank (10) exceeds gallery size (0).

In [ ]:
result_msmt17 = evaluator.evaluate(query_ms, gallery_ms)

print("\nMSMT17 results:")
print(f"  mAP    : {result_msmt17.metrics.map:.1f}%   (paper: ~52.9%)")
print(f"  Rank-1 : {result_msmt17.metrics.rank1:.1f}%   (paper: ~78.7%)")
print(f"  Rank-5 : {result_msmt17.metrics.rank5:.1f}%")
print(f"  Rank-10: {result_msmt17.metrics.rank10:.1f}%")
print(f"  mINP   : {result_msmt17.metrics.minp:.1f}%")

---
## 5. Summary table

In [ ]:
print(f"{'Dataset':<14} {'mAP':>8} {'Rank-1':>8} {'Rank-5':>8} {'Rank-10':>9} {'mINP':>8}")
print("-" * 55)

def _row(name, m):
    return f"{name:<14} {m.map:>7.1f}% {m.rank1:>7.1f}% {m.rank5:>7.1f}% {m.rank10:>8.1f}% {m.minp:>7.1f}%"

print(_row("Market-1501", result_market.metrics))
print(_row("MSMT17",      result_msmt17.metrics))
print("-" * 55)
print("Paper targets: Market-1501 R1≈94.8 mAP≈84.9  |  MSMT17 R1≈78.7 mAP≈52.9")